# Logistic regression and EF

## Introduction
We want to classify data points $X$ using a linear classifier in the feature space $\Phi(X)$.
Here I use linear features, but Gaussian features "blobs" [work as well](https://youtu.be/MS7lz-0E8lM?si=SlwlXF6UeOhmpl34&t=4502).
This means we want to find a linear hyperplane, defined by some weights $w$, that discriminates between two classes here $y=+1$ and $y=-1$.
This is a linear classifier because the decision boundary, defined by the log ratios aka logits, is linear: the decision boundary is when the probability of being of class $+1$ is $p=0.5$, which can be tranlsated into a linear problem in the logits by saying $p=0.5$ implies $a=0$.

Critically, we want to be Bayesian, that is we dont want to get only a point estimate, but have an idea of the confidence of the predictions for some test points $X^*$.
We can't do this in closed form, because of the sigmoid function $\sigma(.)$, which makes the problem non-linear in the probabilities.
The crux of the problem is this probability distribution (classes are discrete and finite, so p is a distribution not density):
$$ p(y= +1 | \Phi(X), w) = \sigma\left(y\,\Phi(X) w\right)$$
which is a complicated function in the space of $w$ and because is the $\sigma(.)$ of a Gaussian distribution $ y \Phi(X) w$ (as we assume that $w$ prior is Gaussian, and the product then is still Gaussian). A sigmoid of a Gaussian is a complicated non-Gaussian distribution and thus finding optimal $w$ given $\Phi(X)$ and then making predictions is complicated. 

However, we can still do something:
1. we can leverage the linearity in the logits to perform a Laplace's approximation + MAcKay's approximation for label prediction
2. MCMC

We could also take the gradient of the negative log-likelihood, but this would only find a point estimate (MLE or MAP) and not the uncertainty.
## Summary
**Workflow.** 
1. define the model
    1. define prior and perform prior predictive checks
    2. define likelihood
4. infer posterior
5. check heuristics
6. perform posterior predictive by push forward

**Laplace approximation.**
TODO

**MCMC heuristics.**
Convert JAX posterior sampled by BlackJax into `arviz` datasets using `from_dict`.
Heuristics
1. Rhat and ESS but need multiple chains
2. energy for HMC, NUTS etc **TODO**
3. pairplots and traces
4. use `arviz.summary`

**Random walk Metropolis didn't work.**
Why? TODO.

In [ ]:
import jax
import numpy as np
import matplotlib.pyplot as plt
from datetime import date
from functools import partial
import jaxstuff
import arviz as az
import jax.numpy as jnp
import blackjax


rnd_state = int(date.today().strftime("%Y%m%d"))
rng_key = jax.random.key(rnd_state)


def plot_predictions(ax, X, P, grid_size):
    assert P.shape[0] == grid_size
    assert P.shape[1] == grid_size
    xx, yy = test_point_coordinates(X, grid_size)
    cf = ax.contourf(
        np.asarray(xx),
        np.asarray(yy),
        np.asarray(P),
        np.arange(0, 1.1, 0.1),
        cmap="RdBu_r",
        vmin=0,
        vmax=1,
    )
    plt.colorbar(
        cf,
        ax=ax,
        label="P(class = +1)",
    )

    # decision boundary at p=0.5
    # ax.contour(np.asarray(xx), np.asarray(yy), np.asarray(P),
    #           levels=[0.5], linewidths=2)

    ax.set_xlabel("x1")
    ax.set_ylabel("x2")


def test_point_coordinates(X, grid_size):
    min_ = X.min(axis=0)
    max_ = X.max(axis=0)
    pad = 0.5
    x1 = jnp.linspace(min_[0] - pad, max_[0] + pad, grid_size)
    x2 = jnp.linspace(min_[1] - pad, max_[1] + pad, grid_size)
    return jnp.meshgrid(x1, x2, indexing="xy")  # (G,G), (G,G)


def test_points(X, grid_size):
    xx, yy = test_point_coordinates(X, grid_size)
    # need to stack and reshape to get a grid_sizex2 array
    return jnp.stack([xx.ravel(), yy.ravel()], axis=1)  # (G*G, 2)

## Prepare the data and constants

In [ ]:
## MCMC
# variance of the proposal distribution (Gaussian) in the Metropolis
# MCMC algorithm. This is proportional to the distance travelled in
# sampling the posterior, but TAU too high results in "sticky" chains.
TAU = 0.005
BURNIN = 500
CHAINS = BURNIN + 1_000

# data
N = 50
X, y = jaxstuff.logistic.make_blobs_two_cls(N, rnd_state)
colors = ["tab:red" if el > 0 else "tab:blue" for el in y]
fig, ax = plt.subplots(figsize=(6, 5), layout="tight")
ax.scatter(*np.asarray(X).T, c=colors, s=30, edgecolors="k")
plt.show()

# features: training and predicting ("testing") features constructed from 2D data
FEAT = "linear"  # "gaussian"
grid_size = 500
Xg = test_points(X, grid_size)
if FEAT == "linear":
    # Dimesions of the problem (dimensions of w not X)
    D = 2 + 1  # need to add the bias term
    # more datapoints then weights as we are in 2D: N > D
    Phi = jaxstuff.features.linear_features(X)
    Phi_grid = jaxstuff.features.linear_features(Xg)
elif FEAT == "gaussian":
    D = 300  # here D < N, so uncertainty
    rng_key, feat_key = jax.random.split(rng_key)
    Phi = jaxstuff.features.gaussian_features(X, D, feat_key)
    rng_key, grid_key = jax.random.split(rng_key)
    Phi_grid = jaxstuff.features.gaussian_features(
        Xg, n_features=D, feat_key=grid_key, ell=0.2
    )  # (G*G, D)
else:
    NotImplementedError(f"Features {FEAT} not implemented")
assert Phi.shape == (N, D)

# prior
PRIOR_SAMPLES = 100
# Variance of the Gaussian prior.
# Smaller sigmas mean higher regularisation, i.e. we give more
# importance to the prior and a bit less to the likelihood (the
# data that is).
SIGMA = 2
logprior_gaussian_sigma = partial(jaxstuff.logistic.logprior_gaussian, sigma=SIGMA)
rng_key, weights_key = jax.random.split(rng_key)
# take some samples from the prior for plotting
w_prior = jax.random.multivariate_normal(
    weights_key, -jnp.ones(D), SIGMA * jnp.eye(D), shape=(PRIOR_SAMPLES,)
)

## The model

### The prior

In [ ]:
# PRIOR PREDICTIVE DISTRIBUTION
# draws of w from the Gaussian prior projected into the
# feature space and shown alongside the data.
p_grid_first_sample = jaxstuff.logistic.push_forward(Phi_grid, w_prior[0, :])  # (G*G,)
p_grid = jaxstuff.logistic.push_forward(Phi_grid, w_prior.T).mean(axis=-1)
# reshape them back for contourf
P_first_sample = p_grid_first_sample.reshape(grid_size, grid_size)
P = p_grid.reshape(grid_size, grid_size)  # (G,G)

fig, ax = plt.subplots(figsize=(6, 5), layout="tight")
plot_predictions(ax, X, P_first_sample, grid_size)
ax.scatter(*np.asarray(X).T, c=colors, s=30, edgecolors="k")
ax.set_title(f"One sample from the prior")
plt.show()

# shows less extreme values thanks to avg
fig, ax = plt.subplots(figsize=(6, 5), layout="tight")
plot_predictions(ax, X, P, grid_size)
ax.scatter(*np.asarray(X).T, c=colors, s=30, edgecolors="k")
ax.set_title(f"{PRIOR_SAMPLES} samples from the prior")
plt.show()

## Finding the mode
We want to find the mode of the posterior distribution.
As the problem is convex/concave the mode will be unique and thus the gradient will eventually converge towrads that value.

The predictions are here one point estimate: each test point is assigned to two numbers (prob of being in class 1 or 2), but there is no uncertainty around these numbers.
Statistical learning, like in deep learning, gives only one point estimate for these probabilities.
### Algo 1: Optimisation with gradient ascent
The less efficient but very simple: leverages `autodiff` for a 1st order method.

In [ ]:
%%time
mode_posterior_gradient = jaxstuff.inference.find_mode_gd(
    w_prior[0, :],
    y,
    Phi,
    log_likelihood=jaxstuff.logistic.loglikelihood,
    log_prior=logprior_gaussian_sigma,
    eps=0.1,
    iterations=10_000,
)
w_gradient = mode_posterior_gradient.inferred_mode.mode

fig, ax = plt.subplots(1, 1, layout="tight")
ax.plot(mode_posterior_gradient.inferred_mode.logdensities)
ax.set_ylabel("Negative posterior density")
ax.set_xlabel("Iterations")
plt.show()

In [ ]:
p_grid = jaxstuff.logistic.push_forward(Phi_grid, w_gradient)
# reshape them back for contourf
P_first_sample = p_grid_first_sample.reshape(grid_size, grid_size)
P = p_grid.reshape(grid_size, grid_size)  # (G,G)

fig, ax = plt.subplots(figsize=(6, 5), layout="tight")
plot_predictions(ax, X, P_first_sample, grid_size)
ax.scatter(*np.asarray(X).T, c=colors, s=30, edgecolors="k")
plt.show()

### Algo 2: Newton's method with autodiff

### Algo 3: Newton's method without autodiff

## Adding uncertainty
### Algo 1: Laplace approximation

### Algo 2: MacKay approximation

### Algo 3: Random-walk Metropolis MCMC
Use a Normal distribution with covariance $\tau^2 I_M$, where $\tau$ is then a "step" size.
The average distance travelled is proportional to $\tau$ and the number of dimensions.

In [ ]:
# scan :: (f, (c, [a])) -> (c, [b])
# f :: (c, a) -> (c, b)
def inference_loop(rng_key, kernel, initial_state, num_samples):
    @jax.jit  # this is not necessary as scan compiles the fn
    def one_step(state, rng_key):  # f :: (c, a) -> (c, b)
        state, info = kernel(rng_key, state)
        return state, (state, info)

    keys = jax.random.split(rng_key, num_samples)
    # (f, (c, [a])) -> (c, [b])
    _, (states, infos) = jax.lax.scan(one_step, initial_state, keys)
    # note that b is a PyTree (per-step output) so JAX will stack each leaf
    # across time, which looks like a transpose from “sequence of pytrees”
    # to “pytree of sequences”. So instead of
    # [(state0, info0), (state1, info1), ...] (a “list/array of tuples”) JAX
    # returns (after stacking): ([state0, state1, ...], [info0, info1, ...])

    return states, infos

In [ ]:
logdensity_mcmc = partial(
    jaxstuff.inference.logdensity,
    log_likelihood=jaxstuff.logistic.loglikelihood,
    log_prior=logprior_gaussian_sigma,
    features=Phi,
    labels=y,
)

In [ ]:
rmh = blackjax.rmh(logdensity_mcmc, blackjax.mcmc.random_walk.normal(jnp.ones(D) * TAU))
initial_state = rmh.init(w_prior[0, :])

In [ ]:
rng_key, sample_key = jax.random.split(rng_key)
states, infos = inference_loop(sample_key, rmh.step, initial_state, CHAINS)
w_rmh = states.position[BURNIN:, :]
print(infos.acceptance_rate.mean())
infos.acceptance_rate

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(12, 2))
for i, axi in enumerate(ax):
    axi.plot(states.position[:, i])
    # as we concat first ones and then data, the first weight is the bias
    axi.set_title(f"$w_{i}$" if i else "b")
    axi.axvline(x=BURNIN, c="tab:red")
plt.show()

p_grid = jaxstuff.logistic.push_forward(Phi_grid, w_rmh.T).mean(axis=-1)
# reshape them back for contourf
P = p_grid.reshape(grid_size, grid_size)  # (G,G)

fig, ax = plt.subplots(figsize=(6, 5), layout="tight")
plot_predictions(ax, X, P, grid_size)
ax.scatter(*np.asarray(X).T, c=colors, s=30, edgecolors="k")
plt.show()

### Algo 4: NUTS

In [ ]:
inverse_mass_matrix = jnp.ones(D)  # TODO
nuts = blackjax.nuts(logdensity_mcmc, TAU, inverse_mass_matrix)
initial_state = nuts.init(w_prior[0, :])

In [ ]:
%%time
rng_key, sample_key = jax.random.split(rng_key)
states, infos = inference_loop(sample_key, nuts.step, initial_state, CHAINS)
w_nuts = states.position[BURNIN:, :]
print(infos.acceptance_rate.mean())
infos.acceptance_rate

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(12, 2))
for i, axi in enumerate(ax):
    axi.plot(states.position[:, i])
    # as we concat first ones and then data, the first weight is the bias
    axi.set_title(f"$w_{i}$" if i else "b")
    axi.axvline(x=BURNIN, c="tab:red")
plt.show()

p_grid = jaxstuff.logistic.push_forward(Phi_grid, w_nuts.T).mean(axis=-1)
# reshape them back for contourf
P = p_grid.reshape(grid_size, grid_size)  # (G,G)

fig, ax = plt.subplots(figsize=(6, 5), layout="tight")
plot_predictions(ax, X, P, grid_size)
ax.scatter(*np.asarray(X).T, c=colors, s=30, edgecolors="k")
plt.show()

In [ ]:
p_grid = jaxstuff.logistic.push_forward(Phi_grid, w_nuts[1, :])
# reshape them back for contourf
P = p_grid.reshape(grid_size, grid_size)  # (G,G)

fig, ax = plt.subplots(figsize=(6, 5), layout="tight")
plot_predictions(ax, X, P, grid_size)
ax.scatter(*np.asarray(X).T, c=colors, s=30, edgecolors="k")
plt.show()

In [ ]:
sample_stats = {
    "acceptance_rate": np.asarray(infos.acceptance_rate)[BURNIN:],
    "diverging": np.asarray(infos.is_divergent)[BURNIN:],
    "energy": np.asarray(infos.energy)[BURNIN:],
}

idata = az.from_dict(
    posterior={w: post for w, post in zip(["b", "w1", "w2"], w_nuts.T)},
    sample_stats=sample_stats,
)

az.plot_trace(idata)
plt.show()
az.plot_pair(
    idata,
    var_names=["w1", "w2"],
    kind="hexbin",
    marginals=True,
    figsize=(11.5, 11.5),
)
plt.show()
az.plot_energy(idata)  # TODO
plt.show()
az.summary(idata)

## References
- BlackJax [notebook](https://blackjax-devs.github.io/sampling-book/models/logistic_regression.html).
- Prob ML videos lecture [15](https://www.youtube.com/watch?v=MS7lz-0E8lM&list=PL05umP7R6ij0hPfU7Yuz8J9WXjlb3MFjm&index=15) and [16](https://www.youtube.com/watch?v=86cnSUh1TBs&list=PL05umP7R6ij0hPfU7Yuz8J9WXjlb3MFjm&index=16)
- Bishop's Book chap 4, mostly chap 4.3.3 and 4.5